# Đánh giá E5 + Reranker — tập test (`test_5000.jsonl`)

Pipeline 2 giai đoạn trên corpus **5000 SP** (`ecommerce.csv`):
1. **Bi-encoder** `e5_base_finetuned_5000` → retrieve top-**n**
2. **Cross-encoder reranker** → xếp hạng lại → top-**k**

**Query:** tự nhiên từ `data/training/test_5000.jsonl` (nhãn = `metadata.product_id`).  
**Metrics:** P@k, R@k, F1@k, MRR@k, NDCG@k + ngưỡng τ (EER, min error).

## 1) Cài thư viện

In [ ]:
!pip -q install "sentence-transformers>=3.0.0" "transformers>=4.40.0" "accelerate>=1.1.0" torch datasets pandas scikit-learn numpy tqdm einops

## 2) Setup

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/PhamMinhDan/llm_provider_benchmarking_ver2.git"
COLAB_REPO = Path("/content/llm_provider_benchmarking")

if COLAB_REPO.exists() and (COLAB_REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(COLAB_REPO), "pull", "--ff-only"], check=False)
elif not (COLAB_REPO / "embedding_project" / "data" / "ecommerce.csv").is_file():
    if COLAB_REPO.exists():
        shutil.rmtree(COLAB_REPO)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO)], check=True)

REPO_DIR = COLAB_REPO
SCRIPTS = REPO_DIR / "embedding_project" / "scripts"
sys.path.insert(0, str(SCRIPTS))

from google.colab import drive
drive.mount("/content/drive")

EMB_MODEL = Path("/content/drive/MyDrive/models/e5_base_finetuned_5000")
RERANKER = Path("/content/drive/MyDrive/models/reranker")
EVAL_CSV = REPO_DIR / "embedding_project/data/ecommerce.csv"
TEST_JSONL = REPO_DIR / "data/training/test_5000.jsonl"
OUTPUT_JSON = REPO_DIR / "embedding_project/outputs/evaluation/reranker_pipeline_eval_test.json"
THRESHOLD_JSON = REPO_DIR / "embedding_project/outputs/evaluation/reranker_threshold_test.json"

for label, p in [("embedding", EMB_MODEL), ("reranker", RERANKER), ("corpus", EVAL_CSV), ("test", TEST_JSONL)]:
    print("OK" if p.exists() else "MISSING", label, "->", p)

## 3) Chạy eval trên tập test

1. Encode E5 → **Recall@n** → chọn `selected_n`
2. Rerank tại `selected_n` → grid **k** (5/10/20)

Dùng `--skip-threshold` để nhanh. Ngưỡng τ → cell **4**.

In [ ]:
import torch

if not TEST_JSONL.is_file():
    raise FileNotFoundError(f"Thiếu {TEST_JSONL}")

MAX_QUERIES = None
TARGET_RECALL = 0.95
N_SEARCH_VALUES = [10, 20, 30, 50, 75, 100]
K_VALUES = [5, 10, 20]
EVAL_K = 10
RERANK_BATCH = 32

cmd = [
    sys.executable,
    str(SCRIPTS / "evaluate_reranker_pipeline.py"),
    "--embedding-model", str(EMB_MODEL),
    "--reranker-model", str(RERANKER),
    "--eval-csv", str(EVAL_CSV),
    "--query-jsonl", str(TEST_JSONL),
    "--output", str(OUTPUT_JSON),
    "--eval-k", str(EVAL_K),
    "--target-recall", str(TARGET_RECALL),
    "--n-values", *[str(n) for n in N_SEARCH_VALUES],
    "--n-search-values", *[str(n) for n in N_SEARCH_VALUES],
    "--k-values", *[str(k) for k in K_VALUES],
    "--rerank-batch-size", str(RERANK_BATCH),
    "--skip-threshold",
]
if MAX_QUERIES:
    cmd.extend(["--max-queries", str(MAX_QUERIES)])

print("CUDA:", torch.cuda.is_available())
print("Test:", TEST_JSONL)
print("Lệnh:", " ".join(cmd))
!{" ".join(cmd)}

## 4) Ngưỡng τ trên tập test

Hard negative từ **top-n E5**. Copy script mới vào `embedding_project/scripts/` nếu vừa cập nhật từ local.

In [ ]:
import importlib
import json
import sys
from argparse import Namespace

import matplotlib.pyplot as plt
import pandas as pd
import torch

N_SEARCH_VALUES = [10, 20, 30, 50, 75, 100]
RERANK_BATCH = 32
ENCODE_BATCH = 128
SCRIPT_PATH = SCRIPTS / "evaluate_reranker_pipeline.py"

if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(SCRIPT_PATH)

import evaluate_reranker_pipeline as erp
importlib.reload(erp)

device = "cuda" if torch.cuda.is_available() else "cpu"
args = Namespace(
    embedding_model=EMB_MODEL,
    reranker_model=RERANKER,
    eval_csv=EVAL_CSV,
    query_jsonl=[TEST_JSONL],
    output=OUTPUT_JSON,
    threshold_output=THRESHOLD_JSON,
    n_search_values=N_SEARCH_VALUES,
    encode_batch_size=ENCODE_BATCH,
    rerank_batch_size=RERANK_BATCH,
    max_queries=None,
    max_neg_per_query=3,
    device=device,
)
print(f"Device: {device} | hard neg pool: E5 top-{max(N_SEARCH_VALUES)}")
erp.run_threshold_only(args, device)

## 5) Xem kết quả

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

EVAL_K = 10

if not OUTPUT_JSON.is_file():
    raise FileNotFoundError(f"Chưa có {OUTPUT_JSON} — chạy cell 3 trước.")

result = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
selected_n = result.get("selected_n")
bi = result[f"bi_encoder_only@{EVAL_K}"]
rk_info = result[f"reranker_best_n@{EVAL_K}"]
rk = rk_info["metrics"]

def row_metrics(stage: str, m: dict) -> dict:
    return {
        "stage": stage,
        f"P@{EVAL_K}": m.get(f"Precision@{EVAL_K}", 0.0),
        f"R@{EVAL_K}": m.get(f"Recall@{EVAL_K}", 0.0),
        f"F1@{EVAL_K}": m.get(f"F1@{EVAL_K}", 0.0),
        f"MRR@{EVAL_K}": m.get(f"MRR@{EVAL_K}", 0.0),
        f"NDCG@{EVAL_K}": m.get(f"NDCG@{EVAL_K}", 0.0),
    }

print(f"=== {result.get('eval_source')} ===")
print(f"{result['n_eval_queries']} queries | corpus {result['corpus_size']} | selected_n={selected_n}")
display(pd.DataFrame([
    row_metrics("Bi-encoder", bi),
    row_metrics(f"Reranker (n={rk_info['n']})", rk),
]))

grid = pd.DataFrame(result["grid_search_n_k"]).sort_values("k")
rows = [{"k": int(r["k"]), "P": r[f"Precision@{int(r['k'])}"], "R": r[f"Recall@{int(r['k'])}"],
         "F1": r[f"F1@{int(r['k'])}"], "NDCG": r[f"NDCG@{int(r['k'])}"]} for _, r in grid.iterrows()]
print("\n=== Metric theo k ===")
display(pd.DataFrame(rows))
print("Recall@n:", json.dumps(result["optimal_n_k"].get("recall_by_n", {}), indent=2))

if not THRESHOLD_JSON.is_file():
    print(f"\nChưa có ngưỡng: {THRESHOLD_JSON} — chạy cell 4.")
else:
    thr_data = json.loads(THRESHOLD_JSON.read_text(encoding="utf-8"))
    deploy = thr_data["deployment_threshold"]
    eer, me = deploy["eer"], deploy["min_error_rate"]
    ta = thr_data["threshold_analysis"]
    print(f"\n=== Ngưỡng | {ta['n_pairs']} cặp (hard neg E5) ===")
    display(pd.DataFrame([
        {"loại": "EER", "τ": eer["threshold"], "FPR": eer["FPR"], "FNR": eer["FNR"], "err": eer["error_rate"]},
        {"loại": "Min error", "τ": me["threshold"], "FPR": me["FPR"], "FNR": me["FNR"], "err": me["error_rate"]},
    ]))

    curve = pd.DataFrame(thr_data.get("threshold_curve", []))
    if len(curve):
        # Important: sort by threshold so matplotlib connects points correctly.
        curve["threshold"] = curve["threshold"].astype(float)
        curve = curve.sort_values("threshold")

        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(curve["threshold"], curve["FPR"], label="FPR")
        ax[0].plot(curve["threshold"], curve["FNR"], label="FNR")
        ax[0].axvline(eer["threshold"], ls="--", color="gray")
        ax[0].legend(); ax[0].set_title("FPR/FNR vs τ")
        ax[1].plot(curve["threshold"], curve["error_rate"], color="crimson")
        ax[1].axvline(me["threshold"], ls="--", color="navy")
        ax[1].set_title("Error rate vs τ")
        plt.tight_layout(); plt.show()

## 6) Ghi chú báo cáo

- **n**: nhỏ nhất đạt Recall ≥ target (mặc định 0.95)
- **k**: số kết quả sau rerank (grid 5/10/20)
- **EER**: τ sao cho FPR(τ) ≈ FNR(τ)
- **Min error**: τ làm (FP+FN) nhỏ nhất
- **Triển khai**: lọc `score_reranker >= τ`